# Hybrid GraphRAG for Personalized E-commerce Recommendations

# Part 1. Executive Summary
## 1-1. Objective
The primary objective of this project is to explore the potential of **Retrieval-Augmented Generation (RAG)** as a dynamic alternative to traditional ML/DL recommendation algorithms (e.g., Collaborative Filtering).

This system implements a **Hybrid GraphRAG** architecture that combines **Vector Search** (Semantic Understanding) with a **Neo4j Knowledge Graph** (Brand Loyalty Signals). It demonstrates how RAG can leverage unstructured review data to overcome the "black-box" limitations of traditional models, providing recommendations that are not only accurate but also context-aware and explainable.

## 1-2. Problem Statement
* **The Limitation of Vector Search:** Standard vector search retrieves textually similar products but lacks the context of individual user preferences, such as brand loyalty or purchase history.
* **Goal:** To improve recommendation accuracy (Hit Rate, NDCG) by injecting personalized graph signals into the retrieval process.

## 1-3. Methodology
* **Data:** Amazon Facial Skincare Reviews Dataset (Metadata & Reviews), a subset of the **Amazon Reviews 2023** dataset.
* **Original Source:** [Amazon Reviews 2023 (Hou et al.)](https://amazon-reviews-2023.github.io/)
* **Tech Stack:** Neo4j (Graph DB), OpenAI (Embeddings/LLM), LangChain, Ragas.
* **Algorithm:**
    * **Cold Start:** Uses purely Vector Search based on review semantics.
    * **Personalized:** Boosts ranking scores dynamically based on the user's purchase history (Log-weighted Graph Score).
* **Evaluation Strategy:**
    * **Anti-Leakage:** Utilized LLM-based paraphrasing to convert raw reviews into generic user queries, preventing direct keyword matching.
    * **Metrics:** Quantitative (Hit Rate@10, NDCG@10) & Qualitative (Ragas Framework with GPT-4o Judge).

## 1-4. Key Results
**"Does adding a Knowledge Graph to Vector Search actually improve recommendation accuracy?"**

This project demonstrates a **Hybrid GraphRAG** architecture that outperforms traditional Vector Search by **+5.5%p**. By injecting user brand loyalty signals (Knowledge Graph) into the retrieval process, the system successfully overcomes the limitations of semantic similarity search, especially for ambiguous user queries.

### 1) Quantitative Results (N=50, Paraphrased Queries)
| Metric | Vector Only (Baseline) | Hybrid GraphRAG (Proposed) | Improvement |
| :--- | :--- | :--- | :--- |
| **Hit Rate@10** | 27.47% | **32.97%** | **+5.49%p** |
| **NDCG@10** | 20.81% | **26.27%** | **+5.46%p** |

> **Impact:** The significant increase in Hit Rate proves that personalization signals (Graph) act as a critical "tie-breaker" when textual similarity alone is insufficient.

### 2) Qualitative Quality (Ragas Eval with GPT-4o)
* **Answer Relevancy (0.85):** Highly relevant advice tailored to specific skin concerns.
* **Faithfulness (0.66):** Effectively combines retrieval context with pre-trained knowledge for persuasive recommendations.
* **Robustness:** Implemented a 'Safe Mode' pipeline to handle API instabilities, achieving an 83.5% evaluation success rate.

## 1-5. Key Takeaways
This project highlights the transformative potential of **RAG (Retrieval-Augmented Generation)** in recommender systems compared to traditional ML/DL approaches.

* **Overcoming the "Black Box":** Unlike Collaborative Filtering (CF) which relies solely on sparse interaction matrices, RAG leverages **unstructured data (reviews/text)** to understand the *semantics* of user needs, providing **explainable recommendations**.
* **Mitigating the Cold Start Problem:** Traditional models struggle when interaction history is scarce. Vector Search (a core component of RAG) enables immediate, relevant recommendations for new products or users by matching semantic intent rather than historical patterns.
* **Dynamic Personalization:** The hybrid approach demonstrates that injecting explicit signals (Knowledge Graph) into implicit vector spaces can effectively fine-tune ranking for individual preferences in real-time.

## 1-6. Limitations & Future Work
While the Hybrid GraphRAG model showed significant improvements over Vector Search, the following areas represent opportunities for further research:

1.  **Benchmark against Traditional Algorithms:**
    * This study focused on comparing *Vector Search vs. Hybrid RAG*. Future iterations should benchmark this architecture against traditional baselines (e.g., Matrix Factorization, LightGCN) to fully quantify the performance trade-offs and ROI of the RAG approach.
2.  **Deepening Domain Knowledge:**
    * The current graph logic relies primarily on **Brand Loyalty**.
    * The recommendation engine could be significantly refined by incorporating more nuanced domain signals such as **Price Elasticity**, **Ingredient Analysis** (crucial for skincare), and specific **Category Purchase Triggers** (e.g., treating toner and serum purchasing patterns differently).

---
*See below for detailed implementation and code execution.*

# Part 2: Technical Implementation
## Step 1. Global Setup & Environment Configuration
- Essential environment setup established for reproducibility.
- Necessary libraries (LangChain, Ragas, Neo4j) imported.
- API keys configured and global Neo4j driver initialized.

In [6]:
import os
import time
import random
import getpass

import numpy as np
import pandas as pd
from tqdm import tqdm

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from openai import OpenAI
from neo4j import GraphDatabase
from ragas import evaluate
from ragas.metrics import (faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity)
from datasets import Dataset

In [53]:
# =====================================================
# Global Configuration & API Keys - Change the api key
# =====================================================

os.environ["OPENAI_API_KEY"] = "(key password here)"

# Model Configurations
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"  # For recommendation generation (Speed/Cost)
JUDGE_MODEL = "gpt-4o"      # For Ragas evaluation (High reasoning capability)

# Initialize Global Clients (Reused throughout the notebook)
client = OpenAI()  # Native client for embeddings/generation
llm_service = ChatOpenAI(model=CHAT_MODEL) # LangChain wrapper for service
llm_judge = ChatOpenAI(model=JUDGE_MODEL)  # LangChain wrapper for evaluation
emb = OpenAIEmbeddings(model=EMBEDDING_MODEL) # LangChain wrapper for embeddings

print(f"✔ Models Configured: Service={CHAT_MODEL}, Judge={JUDGE_MODEL}")

✔ Models Configured: Service=gpt-4o-mini, Judge=gpt-4o


In [52]:
# =====================================================
# Access to Neo4j (Global Driver) - Change the password
# =====================================================
URI = "bolt://172.31.240.1:7687"
USERNAME = "neo4j"
PASSWORD = "(your password here)"

# Initialize Global Driver 
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print("✔ Connected to Neo4j globally.")

✔ Connected to Neo4j globally.


In [11]:
# =======================
# 3. Data Loading Paths
# =======================
BASE_PATH = "./data" 
META_FILE = os.path.join(BASE_PATH, "amazon_face_core_meta.csv")
REVIEW_FILE = os.path.join(BASE_PATH, "amazon_face_core_reviews.csv")

## Step 2. Data ETL & Knowledge Graph Construction
- Static tabular data transformed into a dynamic Knowledge Graph structure.
- Database schema and uniqueness constraints defined.
- User, Product, and Brand nodes constructed from pre-processed metadata and reviews.

In [12]:
# ==========================================
# Helper Functions: import products and reviews
# ==========================================

def init_neo4j_schema(driver):
    """
    Initialize database constraints to ensure data integrity and performance.
    Creates unique constraints on User ID, Product ID, Brand Name, etc.
    """
    print("Initializing Database Schema & Constraints...")
    queries = [
        "CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE",
        "CREATE CONSTRAINT product_id IF NOT EXISTS FOR (p:Product) REQUIRE p.asin IS UNIQUE",
        "CREATE CONSTRAINT review_id IF NOT EXISTS FOR (r:Review) REQUIRE r.id IS UNIQUE",
        "CREATE CONSTRAINT brand_name IF NOT EXISTS FOR (b:Brand) REQUIRE b.name IS UNIQUE",
        "CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE"
    ]
    
    try:
        with driver.session() as session:
            for q in queries:
                session.run(q)
        print("Schema Constraints Created Successfully.")
    except Exception as e:
        print(f"Schema Initialization Warning: {e}")

def import_products(driver):
    """
    Import Product, Brand, and Category nodes from metadata CSV.
    """
    if not os.path.exists(META_FILE):
        print(f"Error: Meta file not found: {META_FILE}")
        return

    print(f"\nImporting Products from: {META_FILE}")
    
    df_meta = pd.read_csv(META_FILE)
    df_meta.fillna({'brand': 'Unknown', 'title': 'No Title', 'price': 0}, inplace=True)
    
    print(f"   - Total Products to Import: {len(df_meta):,}")
    
    # Cypher Query: Create Product, Brand, Category and their relationships
    query = """
    UNWIND $rows AS row
    MERGE (p:Product {asin: row.parent_asin})
    SET p.title = row.title, 
        p.price = toFloat(row.price), 
        p.features = row.features
        
    MERGE (b:Brand {name: row.brand})
    MERGE (p)-[:MADE_BY]->(b)
    
    MERGE (c:Category {name: 'Face'})
    MERGE (p)-[:BELONGS_TO]->(c)
    """
    
    # Batch Processing
    batch_size = 1000
    with driver.session() as session:
        for i in tqdm(range(0, len(df_meta), batch_size), desc="Processing Batches"):
            batch = df_meta.iloc[i:i+batch_size].to_dict('records')
            session.run(query, rows=batch)
            
    print("Product Import Completed.")

def import_reviews(driver):
    """
    Import User and Review nodes, creating relationships between Users, Reviews, and Products.
    """
    if not os.path.exists(REVIEW_FILE):
        print(f"Error: Review file not found: {REVIEW_FILE}")
        return

    print(f"\nImporting Reviews from: {REVIEW_FILE}")
    df_review = pd.read_csv(REVIEW_FILE)
    
    df_review['review_id'] = df_review['user_id'] + '_' + df_review['parent_asin']
    
    print(f"   - Total Reviews to Import: {len(df_review):,}")
    
    # Cypher Query: Create User, Review and link them
    query = """
    UNWIND $rows AS row
    
    MERGE (u:User {user_id: row.user_id})
    
    MERGE (r:Review {id: row.review_id})
    SET r.rating = toInteger(row.rating), 
        r.text = row.text, 
        r.timestamp = row.timestamp, 
        r.helpful_vote = toInteger(row.helpful_vote)
        
    MERGE (u)-[:WROTE]->(r)
    
    WITH r, row
    MATCH (p:Product {asin: row.parent_asin})
    MERGE (r)-[:EVALUATED]->(p)
    """
    
    batch_size = 1000
    with driver.session() as session:
        for i in tqdm(range(0, len(df_review), batch_size), desc="Processing Batches"):
            batch = df_review.iloc[i:i+batch_size].to_dict('records')
            session.run(query, rows=batch)
            
    print("Review Import Completed.")

In [13]:
# ==========================================
# Execution
# ==========================================
init_neo4j_schema(driver)
import_products(driver)
import_reviews(driver)

print("✔ All Neo4j imports completed successfully.")

Initializing Database Schema & Constraints...
Schema Constraints Created Successfully.

Importing Products from: ./amazon_face_core_meta.csv
   - Total Products to Import: 20,082


Processing Batches: 100%|███████████████████████████████████████████████████████████████| 21/21 [00:02<00:00,  9.27it/s]


Product Import Completed.

Importing Reviews from: ./amazon_face_core_reviews.csv
   - Total Reviews to Import: 115,250


Processing Batches: 100%|█████████████████████████████████████████████████████████████| 116/116 [00:06<00:00, 18.04it/s]

Review Import Completed.
✔ All Neo4j imports completed successfully.


## 3. Vector Indexing & Embedding Pipeline
- Pipeline established for semantic search on unstructured text.
- Vector embeddings generated for customer reviews using OpenAI models.
- Neo4j vector index (review_embedding_index) populated for fast similarity retrieval.

In [14]:
# ==========================================
# Vector_indexing: Configuration
# ==========================================

EMBEDDING_DIMENSION = 1536
MAX_EMBEDDING_CHARS = 2000
BATCH_SIZE = 1000  # Adjust based on API rate limits

In [15]:
# ==========================================
# Helper Functions
# ==========================================

def get_reviews_without_embeddings(driver, limit=1000):
    """
    Fetch reviews that do not have an embedding property yet.
    """
    query = """
    MATCH (r:Review)
    WHERE r.embedding IS NULL AND r.text IS NOT NULL
    RETURN r.id AS id, r.text AS text
    LIMIT $limit
    """
    with driver.session() as session:
        result = session.run(query, limit=limit)
        return [{"id": record["id"], "text": record["text"]} for record in result]

def update_review_embeddings(driver, updates):
    """
    Update Review nodes using standard SET query.
    """
    query = """
    UNWIND $updates AS row
    MATCH (r:Review {id: row.id})
    SET r.embedding = row.embedding
    """
    try:
        with driver.session() as session:
            session.run(query, updates=updates)
    except Exception as e:
        print(f"Critical Error updating embeddings: {e}")
        raise e

def create_vector_index(driver):
    """
    Create Vector Index if it doesn't exist.
    """
    index_name = "review_embedding_index"
    print(f"Creating/Verifying Vector Index: {index_name}...")
    
    check_query = "SHOW INDEXES WHERE name = $name"
    with driver.session() as session:
        result = session.run(check_query, name=index_name)
        if result.peek():
            print(f"✔ Index '{index_name}' already exists. Skipping creation.")
            return

    create_query = f"""
    CREATE VECTOR INDEX {index_name} IF NOT EXISTS
    FOR (r:Review)
    ON (r.embedding)
    OPTIONS {{indexConfig: {{
      `vector.dimensions`: 1536,
      `vector.similarity_function`: 'cosine'
    }}}}
    """
    try:
        with driver.session() as session:
            session.run(create_query)
        print(f"✔ Successfully created vector index: {index_name}")
        print("Waiting for index to be online...")
        time.sleep(5)
    except Exception as e:
        print(f"⚠ Failed to create index: {e}")

def generate_embeddings(text_list):
    """
    Generate embeddings using the global OpenAI client.
    """
    try:
        clean_texts = []
        for text in text_list:
            if not isinstance(text, str):
                text = str(text)
            text = text.replace("\n", " ")
            text = text[:MAX_EMBEDDING_CHARS] 
            clean_texts.append(text)

        # Use global 'client' and 'EMBEDDING_MODEL'
        response = client.embeddings.create(
            input=clean_texts,
            model=EMBEDDING_MODEL
        )
        return [data.embedding for data in response.data]

    except Exception as e:
        print(f"OpenAI API Error: {e}")
        return []

In [16]:
# ==========================================
# Execution: Embedding & Indexing
# ==========================================
try:
    print("\nStarting Full Vector Embedding Generation...")
    total_processed = 0
    
    # Use global 'driver' directly
    while True:
        # Fetch batch
        batch = get_reviews_without_embeddings(driver, limit=BATCH_SIZE)
        
        if not batch:
            print("✔ No more reviews to process. All embeddings generated.")
            break
        
        texts = [item["text"] for item in batch]
        ids = [item["id"] for item in batch]
        
        print(f"Generating embeddings for batch of {len(texts)} reviews...")
        start_time = time.time()
        
        # API Call (using global client inside function)
        embeddings = generate_embeddings(texts)
        
        if not embeddings:
            print("Failed to generate embeddings. Stopping process.")
            break
            
        # Update DB
        update_data = [{"id": uid, "embedding": emb} for uid, emb in zip(ids, embeddings)]
        update_review_embeddings(driver, update_data)
        
        total_processed += len(batch)
        elapsed = time.time() - start_time
        print(f"   - Processed {total_processed} reviews successfully. (Batch time: {elapsed:.2f}s)")

    # Create Index after processing
    print("\nBuilding Vector Index...")
    create_vector_index(driver)
    
    print("\n✔ All Embedding Steps Completed Successfully!")

except Exception as e:
    print(f"\nError: {e}")


Starting Full Vector Embedding Generation...
✔ No more reviews to process. All embeddings generated.

Building Vector Index...
Creating/Verifying Vector Index: review_embedding_index...
✔ Index 'review_embedding_index' already exists. Skipping creation.

✔ All Embedding Steps Completed Successfully!


## 4. Hybrid GraphRAG Engine Implementation
- Core algorithm implemented to fuse semantic similarity with domain-specific graph signals.
- run_hybrid_query defined to combine vector scores with log-weighted brand loyalty scores.
- generate_answer implemented to synthesize personalized recommendations using the LLM.

In [17]:
# ====================================================================
# Helper Functions: Dynamic Brand Loyalty Weighting & Hybrid Retrieval
# ====================================================================

def get_embedding(text):
    """
    Generate embedding vector using Global Client.
    """
    clean = str(text).replace("\n", " ")
    # Use global 'client' and 'EMBEDDING_MODEL'
    return client.embeddings.create(
        input=[clean], 
        model=EMBEDDING_MODEL
    ).data[0].embedding
    
def analyze_user_loyalty(driver, user_id):
    """
    Analyze user's brand loyalty based on purchase history.
    Returns: loyalty_score (0.0~1.0), top_brand_name, product_list
    """
    query = """
    MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(p:Product)-[:MADE_BY]->(b:Brand)
    WITH u, p, b
    WITH count(p) AS total_count, 
         b.name AS brand_name, 
         count(b) AS brand_count,
         collect(p.title) AS products
    ORDER BY brand_count DESC
    LIMIT 1
    RETURN total_count, brand_name, brand_count, products
    """
    
    with driver.session() as session:
        result = session.run(query, user_id=user_id).single()
        
        if result:
            total = result['total_count']
            # If purchase history is too small (<3), treat as low loyalty
            if total < 3:
                return 0.1, result['brand_name'], result['products']
            
            score = result['brand_count'] / total
            return score, result['brand_name'], result['products']
            
    return 0.0, None, []

In [64]:
# Dynamic Hybrid Retrieval

def run_hybrid_query(driver, query_embedding, user_id=None, loyalty_score=0.0, k=5, exclude_purchased=False, return_context_string=False):
    """
    Unified Hybrid Search Function.
    - Combines Vector Search + Graph Brand History Boosting.
    - Replaces multiple redundant functions in previous cells.
    
    Args:
        return_context_string (bool): If True, returns formatted strings for RAGAS. If False, returns data dicts.
    """
    # 1. Calculate Dynamic Weight
    brand_weight = loyalty_score * 0.5 
    
    # 2. Construct Cypher Query
    # We inject the WHERE clause dynamically using f-string because Cypher doesn't support conditional blocks well.
    # But we use parameters ($embedding, $weight) for values to ensure safety.
    
    filter_clause = ""
    if exclude_purchased and user_id:
        filter_clause = "WHERE NOT EXISTS { MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(product) }"

    cypher_query = f"""
    // 1. Vector Search
    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)
    YIELD node AS similar_review, score AS vector_score

    // 2. Graph Traversal
    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)
    
    // Optional Filter (Exclude Purchased)
    {filter_clause}

    // 3. Personalization (Check History)
    OPTIONAL MATCH (u:User {{user_id: $user_id}})-[:WROTE]->(:Review)-[:EVALUATED]->(:Product)-[:MADE_BY]->(brand)
    WITH product, brand, similar_review, vector_score, count(u) AS history_count

    // 4. Re-ranking Logic (Dynamic Weight Applied via Parameter)
    WITH product, brand, vector_score, history_count, similar_review,
         (vector_score * (1 + (log(1 + history_count) * $brand_weight))) AS final_score

    ORDER BY final_score DESC
    LIMIT $k
    
    RETURN product.title AS product_name,
           brand.name AS brand_name,
           product.features AS features,
           similar_review.text AS review_text,
           vector_score,
           history_count,
           final_score
    """
    
    with driver.session() as session:
        # Use safe parameters for values
        result = session.run(cypher_query, embedding=query_embedding, user_id=user_id, brand_weight=brand_weight, k=k)
        records = [record.data() for record in result]

    # Return Format 1: String List for RAGAS
    if return_context_string:
        return [f"{r['product_name']} by {r['brand_name']}: {r['review_text']}" for r in records]

    # Return Format 2: Dictionary List for Engine/Demo
    return records

In [19]:
def generate_answer(client, query, context, user_id=None, top_brand=None):
    """
    Generate answer using LLM based on retrieved context.
    """
    system_prompt = """
    You are an expert Personal Shopper AI.
    Recommend products based on the user's query and the retrieved candidates.
    
    Analysis of User Style:
    - If the user has a 'Favorite Brand', acknowledge it but also introduce high-quality alternatives.
    - Explain clearly why each product fits their specific need (based on features/reviews).
    """
    
    context_text = ""
    for i, item in enumerate(context):
        # Handle case where context might be simple strings (RAGAS) or dicts (Engine)
        if isinstance(item, str):
            context_text += f"\n[Candidate #{i+1}] {item}"
        else:
            context_text += f"""
            [Candidate #{i+1}]
            - Product: {item.get('product_name')}
            - Brand: {item.get('brand_name')}
            - Relevance Score: {item.get('final_score'):.4f}
            - User's Brand History: {item.get('history_count')} times purchased
            - Key Review Snippet: "{item.get('review_text', '')[:100]}..."
            """
    
    user_prompt = f"""
    User Query: {query}
    User ID: {user_id}
    User's Favorite Brand: {top_brand if top_brand else "None"}
    
    [Market Data / Context]:
    {context_text}
    """

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

## Step 5. Case Study: Dynamic Ranking Logic

This section demonstrates how the **Hybrid GraphRAG algorithm** prioritizes a user's preferred brand in real-time.

**0) Scenario:** Target User (`AEX...`) searches for a product using a review-based query.</br>
**1) Vector Search (Semantic Cosine Similiarity):** The query matches the product description with a high similarity score (**0.9998**).</br>
**2) Graph Signal (Brand Loyalty):** The system detects a previous purchase history with the brand **'First Aid Beauty'** (`History=1`).</br>
**3) Dynamic Boosting:** A log-weighted boost is applied to the vector score.</br>
* *Calculation:* $0.9998 \times (1 + \text{GraphBoost}) \rightarrow \textbf{1.0690}$</br>
* **Outcome:** The product overtakes higher-ranked competitors to secure **Rank #1**, proving the effectiveness of the personalization logic.

In [20]:
# ==========================================
# Case Study: Qualitative Evaluation
# ==========================================

# Target User ID provided by you
TARGET_USER_ID = "AEXCLMGS3Y7SRW5CMLJBNYI2HBZQ"

def get_user_last_review_text(tx, user_id):
    """
    Fetch the last review written by the user to use as a query.
    """
    query = """
    MATCH (u:User {user_id: $user_id})-[:WROTE]->(r:Review)-[:EVALUATED]->(p:Product)
    RETURN r.text AS text, p.title AS product
    ORDER BY r.timestamp DESC
    LIMIT 1
    """
    result = tx.run(query, user_id=user_id).single()
    return (result['text'], result['product']) if result else (None, None)

In [21]:
# ==========================================
# Execution
# ==========================================
try:
    # Use Global Driver
    with driver.session() as session:
        # Step 1: Fetch Last Review to use as Query
        review_text, last_product = session.execute_read(get_user_last_review_text, TARGET_USER_ID)
        
        if not review_text:
            print("User has no reviews found.")
        else:
            print(f"[Context] Last Purchase: {last_product[:30]}...")
            print(f"[Query] Review Text: \"{review_text[:60]}...\"")
            
            # Step 2: Generate Embedding (Use Global Helper)
            query_vec = get_embedding(review_text)
            
            # Step 3: Run the Unified Hybrid Query
            # Note: To get a final weight of 0.1, we pass loyalty_score=0.2 (0.2 * 0.5 = 0.1)
            print(f"\nExecuting Hybrid Query for User: {TARGET_USER_ID}")
            
            results = run_hybrid_query(
                driver, 
                query_embedding=query_vec, 
                user_id=TARGET_USER_ID, 
                loyalty_score=0.2,  # Adjusts weight to 0.1
                k=10,               # Top 10 results
                exclude_purchased=False # Include purchased items to check rank
            )
            
            print(f"***[Ground Truth] User Actually Bought: {last_product}")

            print("\n[Recommendation Results]")
            
            # Highlight if History > 0 (Boosted)
            print(f"{'Brand':<20} | {'History':<7} | {'VecScore':<9} | {'FinalScore':<10} | {'Product'}")
            print("-" * 80)
            
            for r in results:
                mark = "*" if r['history_count'] > 0 else " "
                print(f"{mark} {r['brand_name'][:18]:<18} | {r['history_count']:<7} | {r['vector_score']:.4f}    | {r['final_score']:.4f}     | {r['product_name'][:30]}...")

except Exception as e:
    print(f"\nError: {e}")

[Context] Last Purchase: First Aid Beauty Facial Radian...
[Query] Review Text: "It soothing.love the way.my skin feels..."

Executing Hybrid Query for User: AEXCLMGS3Y7SRW5CMLJBNYI2HBZQ


Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {}, '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    // 1. Vector Search\n    CALL db.index.vector.queryNodes('review_embedding_index', 30, $embedding)\n    YIELD node AS similar_review, score AS vector_score\n\n    // 2. Graph Traversal\n    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)\n\n    // Optional Filter (Exclude Purchased)\n    \n\n    // 3. Personalization (Check History)\n    OPTIONAL MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(:Product)-[:MADE_BY

***[Ground Truth] User Actually Bought: First Aid Beauty Facial Radiance Pads – Daily Exfoliating Pads with AHA that Help Tone & Brighten Skin – 28 Count

[Recommendation Results]
Brand                | History | VecScore  | FinalScore | Product
--------------------------------------------------------------------------------
* First Aid Beauty   | 1       | 0.9998    | 1.0690     | First Aid Beauty Facial Radian...
* TruSkin Naturals   | 1       | 0.8610    | 0.9207     | TruSkin Vitamin C Serum for Fa...
  Juvenu             | 0       | 0.9069    | 0.9069     | Juvenu Skin Intensive Serum Pr...
  Palmer's           | 0       | 0.9002    | 0.9002     | Palmer's Cocoa Butter Formula ...
  Clinique           | 0       | 0.8910    | 0.8910     | Clinique Dramatically Differen...
  dr.organic         | 0       | 0.8887    | 0.8887     | Dr.Organic Purifying Charcoal ...
  Lancôme            | 0       | 0.8861    | 0.8861     | Absolue Premium ßx Absolute Re...
  Unknown            | 0     

## Step 6. Quantitative Evaluation: Performance Metrics
* Hybrid model benchmarked against a standard Vector Search baseline.
* Synthetic "paraphrased queries" generated to prevent data leakage.
* Hit Rate@10 and NDCG@10 measured on a test set of 50 users:
    * **Hit Rate@10:** Measures whether the target product appears within the top-10 results. (Formula: $1$ if $Target \in Top10$ else $0$)
    * **NDCG@10:** Evaluates ranking quality by prioritizing correct items at higher positions. (Formula: $1 / \log_2(Rank + 1)$)
* **Hybrid GraphRAG outperformed the baseline, demonstrating a +5.49%p improvement in Hit Rate@10 and +5.46%p in NDCG@10.**

In [34]:
# ==========================================
# Test Settings
# ==========================================

SAMPLE_SIZE = 50  
TOP_K = 10        # Top-10 recommendations

In [35]:
# ==========================================
# Helper Functions: Evaluation
# ==========================================

def fetch_test_dataset(driver, sample_size=50):
    """
    Fetch 'Core Users' and their 'Last Review' to create a Ground Truth dataset.
    """
    print(f"Building Test Dataset (Sample: {sample_size})...")
    query = """
    MATCH (u:User)-[:WROTE]->(r:Review)-[:EVALUATED]->(p:Product)
    WITH u, count(r) AS review_count
    WHERE review_count >= 5  // Only Core Users
    WITH u
    ORDER BY rand() LIMIT $limit
    
    MATCH (u)-[:WROTE]->(r:Review)-[:EVALUATED]->(target_p:Product)-[:MADE_BY]->(target_b:Brand)
    WITH u, target_p, target_b, r
    ORDER BY r.timestamp DESC
    WITH u, head(collect(target_p.title)) AS target_product, 
            head(collect(r.text)) AS raw_review_text,
            head(collect(target_b.name)) AS target_brand
            
    MATCH (u)-[:WROTE]->(:Review)-[:EVALUATED]->(p:Product)-[:MADE_BY]->(b:Brand)
    WITH u, target_product, raw_review_text, target_brand, count(p) as total, b, count(b) as b_count
    ORDER BY b_count DESC
    WITH u, target_product, raw_review_text, target_brand, total, head(collect(b.name)) as top_brand, head(collect(b_count)) as top_count
    
    RETURN u.user_id as user_id, 
           target_product, 
           raw_review_text, 
           (toFloat(top_count)/total) as loyalty_score
    """
    with driver.session() as session:
        result = session.run(query, limit=sample_size)
        return [record.data() for record in result]

def generate_synthetic_query(review_text):
    """
    [NEW] Convert a specific product review into a generic user search query.
    This prevents 'Data Leakage' where the review text exactly matches the target document.
    """
    prompt = f"""
    Task: You are a user looking for a skincare product. 
    Based on the review you eventually wrote (below), reconstruct the **Search Query** or **Need Statement** you likely had BEFORE buying the product.
    
    Review: "{review_text[:400]}..."
    
    Rules:
    1. Do NOT mention the specific product name or brand.
    2. Focus on skin concerns (e.g., dry, acne) and desired features (e.g., hydrating, scent-free).
    3. Keep it natural and short (10-15 words).
    4. Example: "I need a gentle toner for my sensitive skin that helps with redness."
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content.strip().replace('"', '')
    except:
        return review_text # Fallback to original if API fails

def run_search(driver, embedding, user_id, loyalty_score, mode='hybrid'):
    """
    Execute search wrapper for evaluation.
    """
    # Determine effective loyalty score
    effective_loyalty = loyalty_score if mode == 'hybrid' else 0.0

    # Reuse the unified search function
    results = run_hybrid_query(
        driver,
        query_embedding=embedding,
        user_id=user_id,
        loyalty_score=effective_loyalty,
        k=TOP_K,
        exclude_purchased=False, 
        return_context_string=False
    )
    return [r['product_name'] for r in results]

def calculate_metrics(recommendations, target_product):
    """Calculate Hit Rate and NDCG."""
    try:
        rank = recommendations.index(target_product) + 1
        hit = 1.0
        ndcg = 1.0 / np.log2(rank + 1)
    except ValueError:
        hit = 0.0
        ndcg = 0.0
    return hit, ndcg

In [36]:
# ==========================================
# Quantitative Evaluation: Execution Loop
# ==========================================

print(f"Connecting to Neo4j at {URI} (Using Global Driver)...")
driver.verify_connectivity()

# Prepare Test Data
test_data = fetch_test_dataset(driver, sample_size=SAMPLE_SIZE)
print(f"Fetched {len(test_data)} raw test cases.")

# Pre-processing: Generate Synthetic Queries
print("\n Generating Synthetic Queries (Paraphrasing) to reduce data leakage...")
print("   (Converting specific reviews into general user needs...)")

for case in tqdm(test_data, desc="Paraphrasing"):
    # Convert raw review -> Generic Search Query
    case['query_text'] = generate_synthetic_query(case['raw_review_text'])

# Show an example of the transformation
print(f"\n[Example Transformation]")
print(f"Original Review: {test_data[0]['raw_review_text'][:80]}...")
print(f"Generated Query: {test_data[0]['query_text']}")
print("-" * 50)

results = {
    'vector': {'hit': [], 'ndcg': []},
    'hybrid': {'hit': [], 'ndcg': []}
}

print("\nStarting Evaluation (Comparing Vector vs. Hybrid)...")

# Evaluation Loop
try:
    for case in tqdm(test_data, desc="Evaluating"):
        user_id = case['user_id']
        target = case['target_product']
        loyalty = case['loyalty_score']
        query_text = case['query_text'] # Uses the NEW synthetic query

        # Use global client inside get_embedding
        query_vec = get_embedding(query_text)

        # Vector Only Search
        recs_vector = run_search(driver, query_vec, user_id, loyalty, mode='vector')
        h_v, n_v = calculate_metrics(recs_vector, target)
        results['vector']['hit'].append(h_v)
        results['vector']['ndcg'].append(n_v)

        # Hybrid Search
        recs_hybrid = run_search(driver, query_vec, user_id, loyalty, mode='hybrid')
        h_h, n_h = calculate_metrics(recs_hybrid, target)
        results['hybrid']['hit'].append(h_h)
        results['hybrid']['ndcg'].append(n_h)
        
    print("\n✔ Evaluation Loop Completed.")

except Exception as e:
    print(f"\nError during evaluation loop: {e}")

Connecting to Neo4j at bolt://172.31.240.1:7687 (Using Global Driver)...
Building Test Dataset (Sample: 50)...
Fetched 91 raw test cases.

🤖 Generating Synthetic Queries (Paraphrasing) to reduce data leakage...
   (Converting specific reviews into general user needs...)


Paraphrasing: 100%|█████████████████████████████████████████████████████████████████████| 91/91 [01:27<00:00,  1.05it/s]



[Example Transformation]
Original Review: This is a great serum when you are like me and like to layer products because it...
Generated Query: Looking for a lightweight, hydrating serum for mature skin that layers easily.
--------------------------------------------------

Starting Evaluation (Comparing Vector vs. Hybrid)...


Evaluating:   0%|                                                                                | 0/91 [00:00<?, ?it/s]Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {}, '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    // 1. Vector Search\n    CALL db.index.vector.queryNodes('review_embedding_index', 30, $embedding)\n    YIELD node AS similar_review, score AS vector_score\n\n    // 2. Graph Traversal\n    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)\n\n    // Optional Filter (Exclude Purchased)\n    \n\n    // 3. Personalization


✔ Evaluation Loop Completed.


In [37]:
# ==========================================
# Quantitative Evaluation: Final Report
# ==========================================

# Calculate Averages
avg_v_hit = np.mean(results['vector']['hit']) * 100
avg_v_ndcg = np.mean(results['vector']['ndcg']) * 100
avg_h_hit = np.mean(results['hybrid']['hit']) * 100
avg_h_ndcg = np.mean(results['hybrid']['ndcg']) * 100

print("\n" + "="*40)
print("FINAL EVALUATION REPORT")
print("="*40)

# Print Table
print(f"{'Metric':<12} | {'Vector Only':<11} | {'GraphRAG (Hybrid)':<17} | {'Improvement'}")
print(f"{'-'*12}|{'-'*13}|{'-'*19}|{'-'*12}")
print(f"{'Hit Rate@10':<12} | {avg_v_hit:6.2f}%     | {avg_h_hit:6.2f}%            | {avg_h_hit - avg_v_hit:+.2f}%p")
print(f"{'NDCG@10':<12} | {avg_v_ndcg:6.2f}%     | {avg_h_ndcg:6.2f}%            | {avg_h_ndcg - avg_v_ndcg:+.2f}%p")
print("="*40)

# Conclusion
if avg_h_hit > avg_v_hit:
    print("Conclusion: GraphRAG outperforms Vector Search.")
else:
    print("Conclusion: Performance is similar. Check brand loyalty distribution.")


FINAL EVALUATION REPORT
Metric       | Vector Only | GraphRAG (Hybrid) | Improvement
------------|-------------|-------------------|------------
Hit Rate@10  |  27.47%     |  32.97%            | +5.49%p
NDCG@10      |  20.81%     |  26.27%            | +5.46%p
Conclusion: GraphRAG outperforms Vector Search.


## Step 7. Qualitative Evaluation: Ragas Quality Assessment
* Quality and factual consistency of AI responses assessed via the Ragas framework (Judge: GPT-4o).
* **High Answer Relevancy (0.85)** achieved, confirming the model's ability to generate advice aligned with specific user needs.
* **Faithfulness (0.66)** observed, reflecting a balance between retrieved context and the model's persuasive generation capabilities.
* **Context Precision (0.24):** Measures if the ground truth is ranked highly in the retrieved context. (Formula: $S_{relevant} / Total_{retrieved}$)
* 'Safe Mode' implemented to robustly handle API timeouts, securing valid results for **83.5% (76/91)** of the test samples.
* Key Result:
  * **Dependency on Initial Retrieval (Recall):** Recommendation failures primarily occurred when **overly generic user queries** caused the target product to be excluded from the initial Vector Search candidate pool (Top-150), thereby bypassing the Hybrid GraphRAG's reranking logic.

In [47]:
# ==========================================
# Qualitative Evaluation: RAGAS Settings
# ==========================================

SAMPLE_SIZE = 50 
TOP_K = 3  # Contexts to retrieve

In [48]:
# ==========================================
# Helper Functions
# ==========================================

def generate_ragas_answer(query, contexts):
    """
    Generate an answer using the Service Model (gpt-4o-mini).
    """
    context_block = "\n".join([f"- {c}" for c in contexts])
    prompt = f"""
    User Query: {query}
    
    Contexts:
    {context_block}
    
    Answer the user query based on the contexts provided. Recommend the best product.
    """
    # Use Global Client (gpt-4o-mini defined in Step 1)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [49]:
# ==========================================
# Execution: Data Generation & Judging
# ==========================================
try:
    print(f"Connecting to Neo4j at {URI} (Using Global Driver)...")
    driver.verify_connectivity()

    # Prepare Test Cases (Reuse existing function and the same logic to fetch 'Core Users')
    test_cases = fetch_test_dataset(driver, sample_size=SAMPLE_SIZE)
    print(f"Prepared {len(test_cases)} raw test cases.")

    # Add Paraphrasing Step (Generate 'query_text' from 'raw_review_text')
    print("\n Generating Synthetic Queries for RAGAS...")
    for case in tqdm(test_cases, desc="Paraphrasing"):
        case['query_text'] = generate_synthetic_query(case['raw_review_text'])

    data_samples = {
        'question': [],
        'answer': [],
        'contexts': [],
        'ground_truth': []
    }

    print("\nGenerating Answers and Contexts...")

    # Generation Loop
    for case in tqdm(test_cases, desc="Generating"):
        query_text = case['query_text']
        if not query_text:
            continue

        # Step A: Embed Query (Use Global Helper)
        query_vec = get_embedding(query_text)
        
        # Step B: Retrieve Contexts
        contexts = run_hybrid_query(
            driver,
            query_embedding=query_vec,
            user_id=case['user_id'],
            loyalty_score=case['loyalty_score'],
            k=TOP_K,
            exclude_purchased=False,
            return_context_string=True  #to get text for LLM context
        )

        # Step C: Generate Answer (Service Model: gpt-4o-mini)
        answer = generate_ragas_answer(query_text, contexts)

        # Step D: Append to Dataset
        data_samples['question'].append(query_text)
        data_samples['answer'].append(answer)
        data_samples['contexts'].append(contexts)
        # Note: ground_truth expects a list of strings in RAGAS v0.1+
        data_samples['ground_truth'].append(case['target_product'])

    # 3. RAGAS Evaluation (The Judging Step)
    print("Running RAGAS Evaluation (Judge: GPT-4o)...")

    dataset = Dataset.from_dict(data_samples)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision
    ]

    # Use 'llm_judge' (GPT-4o) for high-quality evaluation
    results = evaluate(
        dataset,
        metrics=metrics,
        llm=llm_judge, 
        embeddings=emb
    )
    
    print("✔ RAGAS Evaluation Completed.")

except Exception as e:
    print(f"\nError during RAGAS evaluation: {e}")

Connecting to Neo4j at bolt://172.31.240.1:7687 (Using Global Driver)...
Building Test Dataset (Sample: 50)...
Prepared 85 raw test cases.

🤖 Generating Synthetic Queries for RAGAS...


Paraphrasing: 100%|█████████████████████████████████████████████████████████████████████| 85/85 [01:19<00:00,  1.07it/s]



Generating Answers and Contexts...


Generating: 100%|███████████████████████████████████████████████████████████████████████| 85/85 [05:34<00:00,  3.93s/it]


Running RAGAS Evaluation (Judge: GPT-4o)...


Evaluating:   0%|          | 0/255 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

✔ RAGAS Evaluation Completed.


In [62]:
# ==========================================
# Qualitative Evaluation: Report
# ==========================================
import pandas as pd
import numpy as np

print("=" * 40)
print("RAGAS EVALUATION REPORT")
print("=" * 40)

# 1. Convert to Pandas DataFrame
df_results = results.to_pandas()

# ---------------------------------------------------------
# Dynamic Column Detection (Safety Check)
# ---------------------------------------------------------
all_cols = df_results.columns.tolist()
print(f"[System] Detected Columns: {all_cols}")

# Helper to find the actual column name
def get_existing_col(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None

# Identify variable columns
col_question = get_existing_col(['user_input', 'question'], all_cols)
col_answer = get_existing_col(['response', 'answer'], all_cols)
col_contexts = get_existing_col(['retrieved_contexts', 'contexts'], all_cols)
col_truth = get_existing_col(['ground_truth', 'ground_truths'], all_cols)

# Identify available metrics in this run
target_metrics = ['faithfulness', 'answer_relevancy', 'context_precision']
existing_metrics = [m for m in target_metrics if m in all_cols]

# ---------------------------------------------------------
# Data Cleaning & Statistics
# ---------------------------------------------------------
total_rows = len(df_results)

# Define criteria for valid rows (must have key metrics)
# We check existence first to avoid KeyError if a metric failed completely
cleanup_subset = [m for m in ['faithfulness', 'answer_relevancy'] if m in existing_metrics]

if cleanup_subset:
    df_clean = df_results.dropna(subset=cleanup_subset)
else:
    df_clean = df_results # If metrics are missing, keep all (or empty)

clean_count = len(df_clean)
missing_count = total_rows - clean_count

print(f"\n[Data Status]")
print(f"- Total Sample Count: {total_rows}")
print(f"- Analyzable Sample Count: {clean_count} (Success Rate: {clean_count/total_rows*100:.1f}%)")
print(f"- Missing/Failed Sample Count: {missing_count}")

# ---------------------------------------------------------
# Final Scores & Top Examples
# ---------------------------------------------------------
print("-" * 40)
print("[Final Performance Evaluation Results (Mean)]")

if clean_count > 0:
    for metric in existing_metrics:
        score = df_clean[metric].mean()
        print(f" - {metric}: {score:.4f}")
else:
    print(" No valid data available to calculate means.")

print("\n[Top 3 Samples based on Answer Relevancy]")

# Construct display columns
display_cols = []
if col_question: display_cols.append(col_question)
if col_answer: display_cols.append(col_answer)
display_cols.extend(existing_metrics)

# Sort and display using Cleaned Data
if 'answer_relevancy' in df_clean.columns and not df_clean.empty:
    print(df_clean[display_cols].sort_values(by='answer_relevancy', ascending=False).head(3))
elif not df_clean.empty:
    print(df_clean[display_cols].head(3))
else:
    print("No data to display.")

RAGAS EVALUATION REPORT
[System] Detected Columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision']

[Data Status]
- Total Sample Count: 91
- Analyzable Sample Count: 76 (Success Rate: 83.5%)
- Missing/Failed Sample Count: 15
----------------------------------------
[Final Performance Evaluation Results (Mean)]
 - faithfulness: 0.6603
 - answer_relevancy: 0.8492
 - context_precision: 0.2405

[Top 3 Samples based on Answer Relevancy]
                                           user_input  \
76  Looking for a lightweight moisturizer that sme...   
71  Looking for a hydrating balm with good slip fo...   
34  Looking for a hydrating balm with good slip fo...   

                                             response  faithfulness  \
76  Based on your request for a lightweight moistu...      0.363636   
71  Based on your search for a hydrating balm with...      0.533333   
34  Based on your search for a hydrating balm with.

## Step 8. Interactive Demo
* Interactive interface provided for real-time testing via manual input of User IDs and queries.
* RAG retrieval and LLM generation processes observed dynamically.
* System adaptability verified across diverse scenarios, including "Cold Start" cases and specific user contexts.

In [65]:
# ==========================================
# Execution: Interactive Demo
# ==========================================

# Ensure prerequisite steps are run
if 'client' not in locals() or 'driver' not in locals():
    raise ValueError(" Please run the previous 'Global Setup' and 'Helper Functions' cells first.")

print("\nDynamic GraphRAG Engine Initialized! (Ready for Inputs)")

try:
    # Use Global Driver (No need to reconnect)
    driver.verify_connectivity()
    
    while True:
        print("\n" + "="*60)
        # 1. User ID Input
        user_input = input("User ID (Press Enter for Cold Start, 'exit' to quit): ").strip()
        if user_input.lower() == 'exit': 
            print("Exiting Demo. Goodbye! ")
            break
            
        user_id = user_input if user_input else None
        
        # 2. Analyze User Profile
        loyalty_score, top_brand, purchased_list = 0.0, None, []
        if user_id:
            loyalty_score, top_brand, purchased_list = analyze_user_loyalty(driver, user_id)
            print(f"[User Profile] Loyalty Score: {loyalty_score:.2f} (Favorite: {top_brand})")
            print(f"   Already Purchased: {len(purchased_list)} items")

        # 3. Query Input
        query_input = input("Your Question: ").strip()
        if query_input.lower() == 'exit': 
            break
        if not query_input: 
            query_input = "Recommend something based on my taste."
        
        # 4. Set Search Mode
        mode = input("Mode? (1: Discover New, 2: Repurchase/Loyalty) [Default: 1]: ").strip()
        exclude_purchased = False if mode == '2' else True
        
        print(f"\nSearching... (Exclude Purchased: {exclude_purchased})")
        
        # 5. Embed & Search (Unified Function)
        # [FIX] Use global client inside get_embedding (remove 'client' arg)
        query_vec = get_embedding(query_input)
        
        # [FIX] Use 'run_hybrid_query' instead of 'hybrid_search_dynamic'
        retrieved_items = run_hybrid_query(
            driver,
            query_embedding=query_vec,
            user_id=user_id,
            loyalty_score=loyalty_score,
            exclude_purchased=exclude_purchased,
            k=5,
            return_context_string=False  # Return dicts for display
        )
        
        if not retrieved_items:
            print("No products found.")
            continue
            
        # 6. Show Results
        print(f"Top 5 Recommendations:")
        for item in retrieved_items:
            tag = "[Brand Pick]" if item['brand_name'] == top_brand else "✨ [New Discovery]"
            print(f"   - {tag} {item['product_name'][:40]}... (Score: {item['final_score']:.4f})")
        
        # 7. Generate AI Response
        print("\nGenerating Insight...")
        # Note: generate_answer still takes 'client' as per our previous definition
        answer = generate_answer(client, query_input, retrieved_items, user_id, top_brand)
        
        print("\n" + "-"*20 + " AI Response " + "-"*20)
        print(answer)
        print("-" * 60)

except Exception as e:
    print(f"\n Error: {e}")


Dynamic GraphRAG Engine Initialized! (Ready for Inputs)



User ID (Press Enter for Cold Start, 'exit' to quit):  
Your Question:  I need a soothing moisturizer for dry and sensitive skin
Mode? (1: Discover New, 2: Repurchase/Loyalty) [Default: 1]:  1



Searching... (Exclude Purchased: True)


Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {}, '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    // 1. Vector Search\n    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)\n    YIELD node AS similar_review, score AS vector_score\n\n    // 2. Graph Traversal\n    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)\n\n    // Optional Filter (Exclude Purchased)\n    \n\n    // 3. Personalization (Check History)\n    OPTIONAL MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(:Product)-[:MADE_B

Top 5 Recommendations:
   - ✨ [New Discovery] Neutrogena Hydro Boost Hyaluronic Acid H... (Score: 0.8818)
   - ✨ [New Discovery] UGARDEN CICAPLUS Foam Cleanser - Cica an... (Score: 0.8791)
   - ✨ [New Discovery] Pyunkang Yul Calming Intensive Repair Ba... (Score: 0.8714)
   - ✨ [New Discovery] Peptide Collagen Serum for Face - Anti-A... (Score: 0.8653)
   - ✨ [New Discovery] Clinique Moisture Surge 72-Hour Auto-Rep... (Score: 0.8582)

Generating Insight...

-------------------- AI Response --------------------
Based on your need for a soothing moisturizer for dry and sensitive skin, I’ve found several excellent options that cater specifically to these requirements. Here are my top recommendations:

1. **Neutrogena Hydro Boost Hyaluronic Acid Hydrating Water Gel**  
   - **Size:** 1.7 fl. Oz  
   - **Why It Fits:** This moisturizer is oil-free and non-comedogenic, making it an excellent choice for sensitive skin. The key ingredient, hyaluronic acid, is known for its ability to retain mo

...


Searching... (Exclude Purchased: True)


Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {}, '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    // 1. Vector Search\n    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)\n    YIELD node AS similar_review, score AS vector_score\n\n    // 2. Graph Traversal\n    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)\n\n    // Optional Filter (Exclude Purchased)\n    WHERE NOT EXISTS { MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(product) }\n\n    // 3. Personalization (Check History)\n  

Top 5 Recommendations:
   - ✨ [New Discovery] TruSkin B3 Niacinamide Serum for Face, A... (Score: 1.1049)
   - ✨ [New Discovery] Neutrogena Hydro Boost Hyaluronic Acid H... (Score: 0.8818)
   - ✨ [New Discovery] UGARDEN CICAPLUS Foam Cleanser - Cica an... (Score: 0.8791)
   - ✨ [New Discovery] Pyunkang Yul Calming Intensive Repair Ba... (Score: 0.8714)
   - ✨ [New Discovery] Peptide Collagen Serum for Face - Anti-A... (Score: 0.8653)

Generating Insight...

-------------------- AI Response --------------------
Based on your request for a soothing moisturizer for dry and sensitive skin, I see that you have a favorite brand, Dr. Denese. While Dr. Denese products are excellent, I would also like to introduce some high-quality alternatives that align well with your needs.

1. **Neutrogena Hydro Boost Hyaluronic Acid Hydrating Water Gel**
   - **Why it's a great choice:** This moisturizer is specifically designed for dry skin and is oil-free and non-comedogenic, making it suitable for sensi

User ID (Press Enter for Cold Start, 'exit' to quit):  exit


Exiting Demo. Goodbye! 
